<a href="https://colab.research.google.com/github/siddhartha-sai-17/Transformer/blob/main/AI_Story_Generator_using_GPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Project: AI Story Generator using GPT**
## **Dataset**
Use:
[Hugging Face Dataset](https://huggingface.co/datasets/roneneldan/TinyStories)

# **Problem Statement**

A publishing company wants an AI assistant capable of generating children's stories from short prompts.

Build a GPT-based Story Generation System.
```
Input:
Once upon a time
Output:
Once upon a time there was a little rabbit...
```
Tasks:
```
Data Preparation
Tokenization
Embeddings
Positional Encoding
Masked Attention
GPT Decoder
Training
Generation
Evaluation
```

## **Data Preparation**

In [ ]:
from datasets import load_dataset

ds = load_dataset("roneneldan/TinyStories")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…):   0%|          | 0.00/249M [00:00<?, ?B/s]

data/train-00001-of-00004-5852b56a2bd28f(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/train-00002-of-00004-a26307300439e9(…):   0%|          | 0.00/246M [00:00<?, ?B/s]

data/train-00003-of-00004-d243063613e5a0(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/validation-00000-of-00001-869c898b5(…):   0%|          | 0.00/9.99M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

In [ ]:
ds

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2119719
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 21990
    })
})

In [ ]:
df_train = ds["train"].to_pandas()
df_val = ds["validation"].to_pandas()

In [ ]:
df_train.shape, df_val.shape

((2119719, 1), (21990, 1))

In [ ]:
df_train.head()

,text
0,"One day, a little girl named Lily found a need..."
1,"Once upon a time, there was a little car named..."
2,"One day, a little fish named Fin was swimming ..."
3,"Once upon a time, in a land full of trees, the..."
4,"Once upon a time, there was a little girl name..."


In [ ]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2119719 entries, 0 to 2119718
Data columns (total 1 columns):
 #   Column  Dtype 
---  ------  ----- 
 0   text    object
dtypes: object(1)
memory usage: 16.2+ MB


In [ ]:
df_train.isnull().sum()

,0
text,0


In [ ]:
import re
def clean_text(text):
  text = text.lower()
  text = text.replace("\n", " ")
  text = text.replace("\t", " ")
  text = re.sub(r'\s+', ' ', text)
  text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
  return text
df_train['text'] = df_train['text'].apply(clean_text)
df_val['text'] = df_val['text'].apply(clean_text)

# **Tokenization**

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer

In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(df_train['text'])

In [ ]:
vocab_size = len(tokenizer.word_index) + 1
print(f"Vocabulary Size: {vocab_size}")

Vocabulary Size: 58767


In [ ]:
train_sequences = tokenizer.texts_to_sequences(df_train['text'])
val_sequences = tokenizer.texts_to_sequences(df_val['text'])

print(f"Example sequence: {train_sequences[0][:10]}")

Example sequence: [25, 20, 4, 29, 45, 65, 16, 101, 4, 1780]


## **Embeddings and Positional Encoding**

In [ ]:
import numpy as np

def get_positional_encoding(max_seq_len, d_model):
    arg = np.arange(max_seq_len)[:, np.newaxis] / np.power(10000, (2 * (np.arange(d_model)[np.newaxis, :] // 2)) / np.float32(d_model))
    pos_encoding = np.zeros((max_seq_len, d_model))
    pos_encoding[:, 0::2] = np.sin(arg[:, 0::2])
    pos_encoding[:, 1::2] = np.cos(arg[:, 1::2])
    return tf.cast(pos_encoding[np.newaxis, ...], dtype=tf.float32)

class TransformerEmbedding(tf.keras.layers.Layer):
    def __init__(self, vocab_size, d_model, max_seq_len):
        super(TransformerEmbedding, self).__init__()
        self.d_model = d_model
        self.embedding = tf.keras.layers.Embedding(vocab_size, d_model)
        self.pos_encoding = get_positional_encoding(max_seq_len, d_model)

    def call(self, x):
        seq_len = tf.shape(x)[1]
        x = self.embedding(x)
        x *= tf.math.sqrt(tf.cast(self.d_model, tf.float32))
        x += self.pos_encoding[:, :seq_len, :]
        return x

## **Masked Attention and GPT Decoder**

In [ ]:
class DecoderBlock(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads, dff, rate=0.1):
        super(DecoderBlock, self).__init__()
        self.mha = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model)
        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(dff, activation='relu'),
            tf.keras.layers.Dense(d_model)
        ])
        self.layernorm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = tf.keras.layers.Dropout(rate)
        self.dropout2 = tf.keras.layers.Dropout(rate)

    def call(self, x, training, mask):
        attn_output = self.mha(query=x, value=x, key=x, attention_mask=mask, training=training)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(x + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

In [ ]:
def create_look_ahead_mask(size):
    mask = 1 - tf.linalg.band_part(tf.ones((size, size)), -1, 0)
    return mask

class GPT(tf.keras.Model):
    def __init__(self, num_layers, d_model, num_heads, dff, vocab_size, max_seq_len, rate=0.1):
        super(GPT, self).__init__()
        self.embedding = TransformerEmbedding(vocab_size, d_model, max_seq_len)
        self.decoder_layers = [DecoderBlock(d_model, num_heads, dff, rate) for _ in range(num_layers)]
        self.final_layer = tf.keras.layers.Dense(vocab_size)

    def call(self, x, training=False):
        x = tf.convert_to_tensor(x)
        seq_len = tf.shape(x)[1]
        look_ahead_mask = create_look_ahead_mask(seq_len)

        x = self.embedding(x)
        for i in range(len(self.decoder_layers)):
            # Explicitly pass training and mask as keyword arguments
            x = self.decoder_layers[i](x, training=training, mask=look_ahead_mask)

        logits = self.final_layer(x)
        return logits

    def compute_output_shape(self, input_shape):
        return tf.TensorShape([input_shape[0], input_shape[1], vocab_size])

## **Training**

In [ ]:
num_layers = 4
d_model = 128
dff = 512
num_heads = 8
dropout_rate = 0.1
MAX_SEQ_LEN = 100

gpt_model = GPT(num_layers, d_model, num_heads, dff, vocab_size, MAX_SEQ_LEN, dropout_rate)

loss_object = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction='none')

def loss_function(real, pred):
    mask = tf.math.logical_not(tf.math.equal(real, 0))
    loss_ = loss_object(real, pred)
    mask = tf.cast(mask, dtype=loss_.dtype)
    loss_ *= mask
    return tf.reduce_sum(loss_)/tf.reduce_sum(mask)

gpt_model.compile(optimizer='adam', loss=loss_function)

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

def prepare_data(sequences, max_len):
    input_seq = [seq[:-1] for seq in sequences if len(seq) > 1]
    target_seq = [seq[1:] for seq in sequences if len(seq) > 1]

    input_padded = pad_sequences(input_seq, maxlen=max_len, padding='post', truncating='post')
    target_padded = pad_sequences(target_seq, maxlen=max_len, padding='post', truncating='post')

    return input_padded, target_padded

# Using a subset for demonstration to ensure faster training
X_train, y_train = prepare_data(train_sequences[:50000], MAX_SEQ_LEN)
X_val, y_val = prepare_data(val_sequences[:5000], MAX_SEQ_LEN)

print(f"Training data shape: {X_train.shape}")

Training data shape: (49988, 100)


In [ ]:
BATCH_SIZE = 64

# Re-prepare datasets
train_dataset = tf.data.Dataset.from_tensor_slices((X_train.astype('int32'), y_train.astype('int32'))).shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_dataset = tf.data.Dataset.from_tensor_slices((X_val.astype('int32'), y_val.astype('int32'))).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# Instantiate a fresh model with the fixed class definition
gpt_model = GPT(num_layers, d_model, num_heads, dff, vocab_size, MAX_SEQ_LEN, dropout_rate)

# Compile and start training
gpt_model.compile(optimizer='adam', loss=loss_function, jit_compile=False)
gpt_model.fit(train_dataset, epochs=3, validation_data=val_dataset)

Epoch 1/3


/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'decoder_block_4' (of type DecoderBlock) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'decoder_block_5' (of type DecoderBlock) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'decoder_block_6' (of type DecoderBlock) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/usr/lo

## **Story Generation**
Now that the model is trained, we define a function to perform inference. This involves tokenizing a seed prompt and iteratively predicting the next token until we reach the desired length.

In [ ]:
def generate_story(model, tokenizer, start_prompt, max_length=50, temperature=1.0):
    # Clean and tokenize the input prompt
    cleaned_prompt = clean_text(start_prompt)
    input_sequence = tokenizer.texts_to_sequences([cleaned_prompt])[0]

    for _ in range(max_length):
        # Pad the current sequence to match model input shape
        padded_input = pad_sequences([input_sequence], maxlen=MAX_SEQ_LEN, padding='post')

        # Get predictions (ensure training=False for inference)
        logits = model(padded_input, training=False)

        # Focus on the logit for the last actual token in our current sequence
        # Shape of logits is (1, MAX_SEQ_LEN, vocab_size)
        current_pos = len(input_sequence) - 1
        if current_pos >= MAX_SEQ_LEN:
            break

        last_token_logits = logits[0, current_pos, :] / temperature

        # Sample from the distribution
        predicted_id = tf.random.categorical(tf.expand_dims(last_token_logits, 0), num_samples=1)[0, 0].numpy()

        # Stop if we hit padding or if the sequence is full
        if predicted_id == 0:
            break

        input_sequence.append(predicted_id)

    # Convert IDs back to text
    output_text = tokenizer.sequences_to_texts([input_sequence])[0]
    return output_text

# Let's see the model in action
seed_prompt = "once upon a time there was a little"
story = generate_story(gpt_model, tokenizer, seed_prompt, max_length=50)
print(f"Prompt: {seed_prompt}")
print(f"Generated Story:\n{story}")

## **Story Generation**
Now that the model is trained, we define a function to perform inference. This involves tokenizing a seed prompt and iteratively predicting the next token until we reach the desired length.

In [ ]:
def generate_story(model, tokenizer, start_prompt, max_length=50, temperature=1.0):
    # Clean and tokenize the input prompt
    cleaned_prompt = clean_text(start_prompt)
    input_sequence = tokenizer.texts_to_sequences([cleaned_prompt])[0]

    for _ in range(max_length):
        # Pad the current sequence to MAX_SEQ_LEN
        padded_input = pad_sequences([input_sequence], maxlen=MAX_SEQ_LEN, padding='post')

        # Get predictions
        logits = model(padded_input, training=False)

        # Focus on the last token's prediction
        # logits shape: (batch, seq_len, vocab_size)
        last_token_logits = logits[0, len(input_sequence)-1, :] / temperature

        # Sample from the distribution
        predicted_id = tf.random.categorical(tf.expand_dims(last_token_logits, 0), num_samples=1)[0, 0].numpy()

        # Append to sequence
        input_sequence.append(predicted_id)

        # If the model predicts a padding/stop token, we could break (optional)
        if predicted_id == 0:
            break

    # Convert IDs back to text
    output_text = tokenizer.sequences_to_texts([input_sequence])[0]
    return output_text

# Test the generator
prompt = "once upon a time there was a little"
story = generate_story(gpt_model, tokenizer, prompt, max_length=50)
print(f"Prompt: {prompt}")
print(f"Generated Story:\n{story}")

## **Evaluation**
To evaluate the model, we can calculate the **Perplexity**. Perplexity is a common metric in language modeling that measures how well a probability model predicts a sample. A lower perplexity indicates the model is more confident in its predictions.

In [ ]:
### **Evaluation Complete**
The model has been evaluated above. Perplexity and Loss metrics indicate the model's confidence in generating text based on the TinyStories dataset.

## **Evaluation**
To evaluate the model, we can calculate the **Perplexity**. Perplexity is a common metric in language modeling that measures how well a probability distribution or probability model predicts a sample. A lower perplexity indicates the model is more confident in its predictions.

In [ ]:
import math

# Evaluate the model on the validation dataset to get the loss
eval_results = gpt_model.evaluate(val_dataset)

# Calculate perplexity: exp(cross_entropy_loss)
# Note: our loss_function returns the average cross entropy
perplexity = math.exp(eval_results)

print(f"Validation Loss: {eval_results:.4f}")
print(f"Validation Perplexity: {perplexity:.4f}")

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

def prepare_data(sequences, max_len):
    input_seq = [seq[:-1] for seq in sequences if len(seq) > 1]
    target_seq = [seq[1:] for seq in sequences if len(seq) > 1]

    input_padded = pad_sequences(input_seq, maxlen=max_len, padding='post', truncating='post')
    target_padded = pad_sequences(target_seq, maxlen=max_len, padding='post', truncating='post')

    return input_padded, target_padded

X_train, y_train = prepare_data(train_sequences[:50000], MAX_SEQ_LEN)
X_val, y_val = prepare_data(val_sequences[:5000], MAX_SEQ_LEN)

print(f"Training data shape: {X_train.shape}")

In [ ]:
BATCH_SIZE = 64

train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train)).batch(BATCH_SIZE).shuffle(1000)
val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(BATCH_SIZE)

# Training for a few epochs to demonstrate
gpt_model.fit(train_dataset, epochs=2, validation_data=val_dataset)